In [10]:

# ==============================
# 1. IMPORTS
# ==============================
import torch
import re
from transformers import AutoTokenizer, AutoModelForCausalLM

# ==============================
# 2. LOAD MODEL
# ==============================
model_path = "smart_reply_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path).to("cuda")

# ==============================
# 3. GENERATE FUNCTION
# ==============================
def generate_reply(input_text):
    prompt = f"""### Instruction:
Generate exactly 3 short, natural, and different reply suggestions (one per line, no repetition)

### Conversation:
{input_text}

### Response:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.75,
        top_p=0.9,
        repetition_penalty=1.35,
        do_sample=True
    )

    # ==============================
    # 4. DECODE OUTPUT
    # ==============================
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # extract response only
    response = result.split("### Response:")[-1].strip()

    # ==============================
    # 5. CLEAN TEXT
    # ==============================
    lines = [line.strip() for line in response.split("\n") if line.strip()]

    # fallback: split sentences if needed
    if len(lines) < 3:
        lines = re.split(r'[.!?]', response)
        lines = [l.strip() for l in lines if l.strip()]

    # ==============================
    # 6. GRAMMAR FIXES
    # ==============================
    grammar_fix = {
        "Lunching": "Having lunch",
        "I am": "I'm",
        "Do not": "Don't"
    }

    cleaned_lines = []
    for line in lines:
        for k, v in grammar_fix.items():
            line = line.replace(k, v)
        cleaned_lines.append(line)

    lines = cleaned_lines

    # ==============================
    # 7. ENSURE EXACTLY 3 OUTPUTS
    # ==============================
    lines = lines[:3]

    while len(lines) < 3:
        lines.append("Okay")   # fallback safe reply

    # ==============================
    # 8. REMOVE DUPLICATES
    # ==============================
    unique_lines = []
    for line in lines:
        if line not in unique_lines:
            unique_lines.append(line)

    while len(unique_lines) < 3:
        unique_lines.append("Alright")

    lines = unique_lines[:3]

    # ==============================
    # 9. ADD NUMBERING
    # ==============================
    formatted = "\n".join([f"{i+1}. {line}" for i, line in enumerate(lines)])

    return formatted

In [21]:

# ==============================
# 10. TEST CASES
# ==============================
# print("---- Test 1 ----")
# print(generate_reply("Friend: Are you coming today?"))

print("\n---- Test 2 ----")
print(generate_reply("Friend: What are you doing?"))

# print("\n---- Test 3 ----")
# print(generate_reply("Friend: Want to hang out tonight?"))

# print("\n---- Test 4 ----")
# print(generate_reply("Friend: Why didn't you reply earlier?"))

# print("\n---- Test 5 ----")
# print(generate_reply("Friend: I'm feeling really tired today"))


---- Test 2 ----
1. Going for a walk now
2. Just grabbed some coffee
3. Waiting outside the house
